In [2]:
import os
import numpy as np
import random
from scipy.interpolate import interp1d


"""
Blue Straggler Population Synthesis

This script simulates N binary systems and computes how many
blue stragglers exist at a given time t.

Author: Emilie Desrochers Karlsson
"""


# -------------------------------
# PARAMETERS
# -------------------------------

Z = 0.02  #metallicity 
BS_THRESHOLD = 1.01  # mass threshold for blue straggler


# -------------------------------
# FUNCTIONS
# -------------------------------

def run_bse(m1, m2, t_max, period, ecc, z):
    """
    Runs BSE simulation for a binary system.
    """
    with open("binary.in", "r") as f:
        lines = f.readlines()

    lines[0] = "{} {} {} {} {} {} {} {}\n".format(
        m1, m2, t_max, period, 1, 1, z, ecc
    )

    with open("binary.in", "w") as f:
        f.writelines(lines)

    os.system("./bse")

    try:
        data = np.loadtxt("binary.dat")
        return np.stack(data, axis=-1)
    except ValueError:
        # BSE failed (invalid output)
        return None


def main_sequence_lifetime(m1, z):
    """
    Computes MS lifetime using BSE.
    """
    data = run_bse(m1, 0, 90000, 0, 0.5, z)
    
    if data is None:
        return []

    time = data[0]
    kw = data[1]

    ms_indices = [i for i in range(len(kw)) if kw[i] == 1]
    return time[ms_indices[-1]]


def compute_turnoff_mass(z):
    """
    Computes turnoff mass as function of time.
    """
    masses = np.linspace(0.8, 10, 100)
    lifetimes = [main_sequence_lifetime(m, z) for m in masses]

    return interp1d(lifetimes, masses, kind="cubic", fill_value="extrapolate")


# -------------------------------
# BLUE STRAGGLER CHECK
# -------------------------------

def is_blue_straggler(data, t, turnoff_mass):
    """
    Checks if either star is a blue straggler at time t.
    Returns mass ratio(s) if true.
    """
    
    if data is None:
        return []
    time = data[0]
    kw1, kw2 = data[1], data[2]
    mass1, mass2 = data[3], data[4]

    results = []

    # use last timestep before t
    i = -2

    if kw1[i] == 1 and mass1[i] > BS_THRESHOLD * turnoff_mass(t) and time[i] == t:
        results.append(mass1[i] / turnoff_mass(t))

    if kw2[i] == 1 and mass2[i] > BS_THRESHOLD * turnoff_mass(t) and time[i] == t:
        results.append(mass2[i] / turnoff_mass(t))

    return results


# -------------------------------
# SAMPLING INITIAL PARAMETERS
# -------------------------------

def sample_primary_mass():
    """
    Samples from IMF ~ m^-2.3 (approx implementation).
    """
    return (random.random() * (100**(-1.3) - 0.8**(-1.3)) + 0.8**(-1.3))**(1 / -1.3)


def sample_mass_ratio():
    """
    Uniform mass ratio distribution.
    """
    return random.uniform(0.1, 1)


def sample_period():
    """
    Log-normal period distribution.
    """
    return 10 ** random.gauss(5.03, 2.28)


def sample_eccentricity():
    return random.random()


# -------------------------------
# MAIN SIMULATION
# -------------------------------

def run_population(N, t_max, z):
    """
    Evolves N binaries and counts blue stragglers.
    """
    turnoff_mass = compute_turnoff_mass(z)

    bs_ratios = []

    for _ in range(N):
        m1 = sample_primary_mass()
        q = sample_mass_ratio()
        m2 = m1 / q

        period = sample_period()
        ecc = sample_eccentricity()

        data = run_bse(m1, m2, t_max, period, ecc, z)

        result = is_blue_straggler(data, t_max, turnoff_mass)

        if result:
            bs_ratios.extend(result)

    return bs_ratios


# -------------------------------
# EXECUTION
# -------------------------------

if __name__ == "__main__":

    N = 10000  
    t_max = 8000  # Myr

    bs_ratios = run_population(N, t_max, Z)

    print(f"Number of blue stragglers at t={t_max} Myr: {len(bs_ratios)}")
    print("Mass ratios (M_BS / M_turnoff):")
    print(bs_ratios)

Number of blue stragglers at t=8000 Myr: 2
Mass ratios (M_BS / M_turnoff):
[1.0126389745528694, 1.037362144289185]
